# Neuromod Lab — Colab

Laboratório de arquiteturas de memória: táxis com corpo e economia num grafo urbano.

**Auto-contido** — este notebook escreve o código sozinho; não precisa de zip.

Runtime → Alterar tipo de ambiente → **GPU (T4)**.

Novidades desta versão (2026-08-31):
- **~3× mais rápido** (torch.compile + loop sem syncs CPU↔GPU + janela diádica fundida)
- **Salva por etapas**: resultado PARCIAL no Drive após CADA arquitetura + checkpoint da
  evolução a cada 10 gerações. Sessão caiu? Rode a célula da bateria de novo — ela
  **retoma de onde parou** (pula arquiteturas prontas, retoma a evolução do checkpoint).

| célula | o quê |
|---|---|
| 1 | setup + Drive (resultados vão DIRETO pro Drive) |
| 2–3 | escreve `src/lab.py` e `src/rodar.py` |
| 4 | portão de validação (obrigatório) |
| 5 | bateria 3 sementes |
| 6 | leitura rápida |


In [ ]:
import os, torch
os.makedirs('src', exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA — troque o runtime')

# Drive montado ANTES da bateria: parciais e checkpoints caem la na hora.
# Se a sessao morrer, nada se perde.
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/neuromod-lab-resultados'
os.makedirs(OUT, exist_ok=True)
print('saida:', OUT)

In [ ]:
%%writefile src/lab.py
import torch, numpy as np, json, time, os
from datetime import datetime
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

GMAX = 5    # grau maximo de um no
NPX  = 6    # passageiros disponiveis por mundo

# ============================ CIDADES ============================
def gerar_cidade(seed, n=26):
    r = np.random.default_rng(seed); pos = []
    while len(pos) < n:
        p = r.random(2)*12
        if all(np.hypot(p[0]-q[0], p[1]-q[1]) > 1.2 for q in pos): pos.append(p)
    pos = np.array(pos)
    D = np.hypot(pos[:,None,0]-pos[None,:,0], pos[:,None,1]-pos[None,:,1])
    np.fill_diagonal(D, 1e9)
    adj = [set() for _ in range(n)]
    for i in range(n):
        for j in np.argsort(D[i])[:int(r.integers(2,5))]:
            adj[i].add(int(j)); adj[int(j)].add(i)
    for i in range(n):
        if not adj[i]:
            j = int(np.argmin(D[i])); adj[i].add(j); adj[j].add(i)
    adj = [sorted(a)[:GMAX] for a in adj]
    A = np.full((n,GMAX), -1, dtype=np.int64)
    W = np.zeros((n,GMAX), dtype=np.float32)
    for i in range(n):
        for k, j in enumerate(adj[i]):
            A[i,k] = j; W[i,k] = max(1, round(D[i,j]/1.5))
    DI = np.full((n,n), 1e6, dtype=np.float32); np.fill_diagonal(DI, 0)
    for i in range(n):
        for k, j in enumerate(adj[i]): DI[i,j] = W[i,k]
    for k in range(n): DI = np.minimum(DI, DI[:,k:k+1] + DI[k:k+1,:])
    idx = list(r.permutation(n))
    serv = np.zeros((n,5), dtype=np.float32)   # posto, comida, descanso, oficina, loja
    for t, (a,b) in enumerate([(0,3),(3,6),(6,8),(8,12),(12,14)]):
        for i in idx[a:b]: serv[i,t] = 1.0
    grp = np.zeros((n,GMAX), dtype=np.int64); tem = np.zeros(n, dtype=np.float32)
    for i in range(n):
        if len(adj[i]) >= 3:
            tem[i] = 1.0
            for k, j in enumerate(adj[i]):
                ang = np.arctan2(*(pos[j]-pos[i])[::-1]) % np.pi
                grp[i,k] = 0 if ang < np.pi/2 else 1
    return dict(n=n, A=A, W=W, DI=DI, serv=serv, grp=grp, tem=tem)

class Cfg:
    def __init__(s, d=1.0, trafego=.35, imposto=.18, mundo='base', raio=4.,
                 deriva=.02, per_mare=128):
        # mundo: o que fica OCULTO do taxi.
        #   base    visao global, spawn uniforme      -> memoria nao tem o que pagar
        #   cego    visao raio R, spawn uniforme      -> CONTROLE: oculto sem estrutura
        #   hotspot visao raio R, polo que DERIVA     -> memoria de onde apareceu paga
        #   mare    visao raio R, 2 polos alternando  -> estrutura PERIODICA (fase oculta)
        #   ruas    congestao CRONICA por no (sorteada POR MUNDO, nao por cidade —
        #           evolucao nao decora) e INVISIVEL nos vizinhos: so sente ao pisar.
        #           Lembrar quais ruas entopem = MemCusto. Resto igual ao base.
        #   semgps  oraculo de distancia-ate-servico REMOVIDO (features ds e leitura
        #           do corpo/hibrido ficam neutras). Saber onde tem posto = ter
        #           passado la = MapaServico. Resto igual ao base.
        s.mundo=mundo; s.raio=raio; s.deriva=deriva; s.per_mare=per_mare
        s.cron_frac=.30; s.cron_alto=1.6   # ruas: fracao cronica e teto do custo extra
        s.clarividente=False   # diagnostico: ruas com congestao VISIVEL = teto fisico
        s.fuel=.70+1.4*d; s.pneu=.18+.9*d; s.temp=1.6+4.*d; s.fome=.07+.25*d
        s.fadiga=.45+1.2*d; s.risco=.02+.12*d; s.oleo=.25+.7*d
        s.per=max(int(round(16-6*d)),6); s.base=6.; s.km=2.6
        s.trafego=trafego; s.imposto=imposto

# ============================ MUNDO EM LOTE ============================
# REGRA DE PERF (esta reescrita rendeu ~10x; nao regredir):
#   1. NENHUM sync GPU->CPU no loop quente: nada de `if tensor.any():`,
#      `.item()`, `.nonzero()` — toda ramificacao vira mascara.
#   2. NENHUM tensor novo de constante por passo (torch.tensor(...) e H2D):
#      constantes vivem em __init__.
#   3. Contadores sao tensores no device (int python vira guard/recompile
#      no torch.compile).
#   4. Amostragem categorica por Gumbel-argmax (multinomial faz sync).
class Lote:
    """B = P genomas x NC cidades x R replicas, tudo num tensor so."""
    def __init__(s, cidades, P, cfg, seed=0, R=1):
        s.cfg = cfg
        NC = len(cidades); s.NC, s.P, s.R = NC, P, R
        n = cidades[0]['n']; s.n = n
        s.B = B = P*NC*R
        st = lambda k, dt=torch.float32: torch.stack(
            [torch.tensor(c[k], device=DEV, dtype=dt) for c in cidades])
        s.Aa=st('A',torch.long); s.Wa=st('W'); s.DIa=st('DI')
        s.serva=st('serv'); s.grpa=st('grp',torch.long); s.tema=st('tem')
        ds = []
        for c in range(NC):
            linha = []
            for j in range(5):
                m = cidades[c]['serv'][:,j] > 0
                linha.append(s.DIa[c][:, torch.tensor(m, device=DEV)].min(1).values
                             if m.any() else torch.full((n,), 20., device=DEV))
            ds.append(torch.stack(linha))
        s.dsa = torch.stack(ds)                       # (NC,5,n)
        if cfg.mundo == 'semgps':
            # oraculo removido NA FONTE: features ds e a leitura direta do
            # corpo/hibrido enxergam a mesma constante (ninguem trapaceia)
            s.dsa = torch.full_like(s.dsa, 20.)
        # mundo b pertence ao genoma gidx[b] e a cidade cidx[b]
        s.cidx = torch.arange(NC, device=DEV).repeat_interleave(R).repeat(P)
        s.gidx = torch.arange(P, device=DEV).repeat_interleave(NC*R)
        g = torch.Generator(device=DEV); g.manual_seed(seed); s.g = g
        s.gsim = torch.Generator(device=DEV); s.gsim.manual_seed(seed+999983)
        ri = lambda hi, sh: torch.randint(0, hi, sh, generator=g, device=DEV)
        s.no=ri(n,(B,)); s.indo=torch.full((B,),-1,dtype=torch.long,device=DEV)
        s.rest=torch.zeros(B,device=DEV); s.tpar=torch.zeros(B,device=DEV)
        s.fuel=torch.full((B,),85.,device=DEV); s.fome=torch.full((B,),10.,device=DEV)
        s.fad=torch.full((B,),10.,device=DEV);  s.pneu=torch.full((B,),100.,device=DEV)
        s.temp=torch.full((B,),25.,device=DEV); s.oleo=torch.full((B,),100.,device=DEV)
        s.dinh=torch.full((B,),60.,device=DEV); s.up=torch.zeros(B,9,device=DEV)
        s.po_=torch.full((B,),-1,dtype=torch.long,device=DEV)
        s.pd_=torch.full((B,),-1,dtype=torch.long,device=DEV)
        s.ent=torch.zeros(B,device=DEV); s.rec=torch.zeros(B,device=DEV)
        s.imp=torch.zeros(B,device=DEV); s.avar=torch.zeros(B,device=DEV)
        s.melh=torch.zeros(B,device=DEV)
        s.fase=ri(2,(B,n)).float(); s.tsem=ri(cfg.per,(B,n)).float()
        if cfg.mundo == 'ruas':
            # nos CRONICOS por mundo: rua que sempre entope. Por mundo (nao por
            # cidade) de proposito: evolucao entre episodios nao tem como decorar
            # — so memoria DENTRO da vida resolve.
            cron = (torch.rand(B,n,generator=g,device=DEV) < cfg.cron_frac).float()
            s.npc_base = cron*(cfg.cron_alto*(0.6+0.4*torch.rand(B,n,generator=g,device=DEV))) \
                         + (1-cron)*0.08
            s.npc = s.npc_base.clone()
        else:
            s.npc_base = None
            s.npc=torch.rand(B,n,generator=g,device=DEV)*cfg.trafego*2
        s.po=ri(n,(B,NPX)); s.pd=ri(n,(B,NPX))
        s.t=torch.zeros((),dtype=torch.long,device=DEV)
        # constantes cacheadas (regra 2)
        s.ARB   = torch.arange(B, device=DEV)
        s.AR5   = torch.arange(5, device=DEV)
        s.CUSTO = torch.tensor([2.,3.,0.,10.,26.], device=DEV)
        s.MP    = torch.tensor([0,1,2,3,3], device=DEV)
        s.UNS   = torch.ones(B, GMAX, 1, device=DEV)
        # estado OCULTO do mundo (o taxi nunca ve estes tensores)
        if cfg.mundo in ('hotspot','mare'):
            s.hot = ri(n,(B,))                      # polo atual
            if cfg.mundo == 'mare':
                s.polo2 = ri(n,(B,))                # segundo polo
                s.tmare = ri(cfg.per_mare,(B,)).float()
            # respawn ja concentrado no polo desde o inicio
            for k in range(NPX):
                s.po[:,k] = s._spawn_polo()
    def _spawn_polo(s):
        """(B,) sorteio de no ~ exp(-dist ao polo), para TODOS os mundos de uma
        vez. Gumbel-argmax == softmax-sampling, sem multinomial (que faz sync)."""
        d = s.DIa[s.cidx[:,None].expand(-1,s.n),
                  s.hot[:,None].expand(-1,s.n),
                  torch.arange(s.n, device=DEV)[None,:].expand(s.B,-1)]
        u = torch.rand(s.B, s.n, generator=s.gsim, device=DEV).clamp(min=1e-20)
        return (-d/3. - torch.log(-torch.log(u))).argmax(1)

    # resolve "qual cidade" por mundo
    def A_(s):    return s.Aa[s.cidx, s.no]
    def W_(s):    return s.Wa[s.cidx, s.no]
    def grp_(s):  return s.grpa[s.cidx, s.no]
    def tem_(s):  return s.tema[s.cidx, s.no]
    def serv_(s): return s.serva[s.cidx, s.no]
    def serv_need(s):
        """servico disponivel PONDERADO pela necessidade (posto so importa se falta combustivel)"""
        sv = s.serva[s.cidx, s.no]
        nec = torch.stack([1-s.fuel/s.capf(), s.fome/100., s.fad/100.,
                           torch.maximum(1-s.pneu/100., s.temp/120.),
                           (s.dinh/90.).clamp(max=1.)], -1)
        paga = (s.dinh[:,None] >= s.CUSTO[None,:]).float()
        return sv*nec*paga, sv*paga
    def capf(s): return 100+35*s.up[:,0]
    def txp(s):  return 1./((1+.7*s.up[:,1])*(1+.5*s.up[:,7]))
    def txt(s):  return 1./(1+.7*s.up[:,2])
    def txo(s):  return 1./(1+.8*s.up[:,8])
    def vel(s):  return 1./(1+.3*s.up[:,3])
    def tar(s):  return 1.+.28*s.up[:,4]
    def obs(s):
        B, G = s.B, GMAX
        viz = s.A_(); val = viz >= 0; vz = viz.clamp(min=0)
        ce = s.cidx[:,None].expand(-1,G)
        # passageiro livre mais proximo
        dpo = s.DIa[s.cidx[:,None].expand(-1,NPX), s.no[:,None].expand(-1,NPX), s.po]
        VIS_LIMITADA = s.cfg.mundo in ('cego','hotspot','mare')
        if VIS_LIMITADA:
            # VISAO LIMITADA: passageiro alem do raio nao existe para o taxi.
            # (ruas/semgps mantem visao global: cada mundo esconde UMA coisa so)
            dpo = torch.where(dpo <= s.cfg.raio, dpo, torch.full_like(dpo, 1e6))
        kb = dpo.argmin(1)
        po_alvo = s.po.gather(1, kb[:,None]).squeeze(1)
        alvo = torch.where(s.pd_>=0, s.pd_, po_alvo)
        dal = s.DIa[ce, vz, alvo[:,None].expand(-1,G)]
        if VIS_LIMITADA:
            # sem passageiro visivel e sem corrida: distancia CONSTANTE em todos
            # os vizinhos = zero informacao direcional. E o que abre espaco para
            # a memoria pagar (ou nao — mundo cego e o controle).
            ve = (dpo.min(1).values < 1e5) | (s.pd_ >= 0)
            dal = torch.where(ve[:,None], dal, torch.full_like(dal, 2*s.cfg.raio))
        cus = s.W_()
        if s.cfg.mundo == 'ruas' and not s.cfg.clarividente:
            # congestao dos vizinhos INVISIVEL: so se sente ao pisar no no.
            # A feature fica (I8), mas carrega zero informacao.
            cong = torch.zeros_like(cus)
        else:
            cong = torch.gather(s.npc, 1, vz)
        gs = s.grp_(); fn = torch.gather(s.fase, 1, s.no[:,None]).expand(-1,G)
        vd = ((gs.float()==fn) | (s.tem_()[:,None]==0)).float()*2-1
        ds = s.dsa[s.cidx[:,None,None].expand(-1,5,G),
                   s.AR5[None,:,None].expand(B,-1,G),
                   vz[:,None,:].expand(-1,5,-1)]              # (B,5,G)
        if s.cfg.mundo == 'semgps':
            # feature ZERADA (nao constante!): constante desloca movimento vs
            # servico e paralisa o perito — zero e contribuicao exata nula.
            ds = torch.zeros_like(ds)
        est = torch.stack([s.fuel/s.capf(), s.fome/100, s.fad/100, s.pneu/100,
                           s.temp/120, s.oleo/100, (s.dinh/120).clamp(max=1.),
                           (s.pd_>=0).float()], -1)
        F = torch.cat([(-dal/6.)[...,None], (-cus/4.)[...,None], (-cong/2.)[...,None],
                       vd[...,None], (-ds.permute(0,2,1)/6.),
                       est[:,None,:].expand(-1,G,-1),
                       s.UNS], -1)
        return F, val

NF = 1+1+1+1+5+8+1   # 18 features por candidato (ver NOMES_F em rodar.py)

# ============================ PASSO ============================
def passo(m, esc):
    """esc: (B,) 0..GMAX-1 = vizinho | GMAX..GMAX+4 = servico
    Tudo mascarado, zero ramificacao dependente de dado (regra 1)."""
    c = m.cfg; B = m.B; m.t += 1
    m.tsem += 1; v = m.tsem >= c.per
    m.fase = torch.where(v, 1-m.fase, m.fase)
    m.tsem = torch.where(v, torch.zeros_like(m.tsem), m.tsem)
    m.temp = (m.temp-1.6).clamp(min=25.)
    # entrega
    ch = (m.pd_>=0) & (m.no==m.pd_) & (m.rest<=0)
    d = m.DIa[m.cidx, m.po_.clamp(min=0), m.pd_.clamp(min=0)]
    br = (c.base+c.km*d)*m.tar(); im = br*c.imposto
    m.ent += ch.float(); m.rec += br*ch.float(); m.imp += im*ch.float()
    m.dinh += (br-im)*ch.float()
    m.po_ = torch.where(ch, torch.full_like(m.po_,-1), m.po_)
    m.pd_ = torch.where(ch, torch.full_like(m.pd_,-1), m.pd_)
    livre = m.tpar<=0; m.tpar = (m.tpar-1).clamp(min=0)
    an_ = (m.rest>0) & livre; m.rest = torch.where(an_, m.rest-1, m.rest)
    cg = an_ & (m.rest<=0)
    m.no = torch.where(cg, m.indo.clamp(min=0), m.no)
    m.indo = torch.where(cg, torch.full_like(m.indo,-1), m.indo)
    m.fome = (m.fome+c.fome).clamp(max=100)
    ap = m.fad>=95
    m.tpar = torch.where(ap, torch.full_like(m.tpar,12.), m.tpar)
    m.fad  = torch.where(ap, torch.full_like(m.fad,40.), m.fad)
    pode = livre & (m.rest<=0) & (~ap)
    sv = m.serv_()
    # servicos (sem closures: compile-friendly)
    o1 = pode & (esc==GMAX+0) & (sv[:,0]>0) & (m.dinh>=2.)
    m.dinh = m.dinh - 2.*o1.float()
    m.fuel = torch.where(o1, torch.minimum(m.fuel+30, m.capf()), m.fuel)
    o2 = pode & (esc==GMAX+1) & (sv[:,1]>0) & (m.dinh>=3.)
    m.dinh = m.dinh - 3.*o2.float()
    m.fome = torch.where(o2, (m.fome-45).clamp(min=0), m.fome)
    o3 = pode & (esc==GMAX+2) & (sv[:,2]>0)
    m.fad = torch.where(o3, (m.fad-30).clamp(min=0), m.fad)
    o4 = pode & (esc==GMAX+3) & (sv[:,3]>0) & (m.dinh>=10.)
    m.dinh = m.dinh - 10.*o4.float()
    m.pneu = torch.where(o4, torch.full_like(m.pneu,100.), m.pneu)
    m.oleo = torch.where(o4, torch.full_like(m.oleo,100.), m.oleo)
    m.temp = torch.where(o4, torch.full_like(m.temp,25.), m.temp)
    o5 = pode & (esc==GMAX+4) & (sv[:,4]>0) & (m.dinh>=26.)
    m.dinh = m.dinh - 26.*o5.float()
    k = (m.up.sum(1) % 9).long()
    m.up[m.ARB, k] += o5.float(); m.melh += o5.float()
    serv_ok = o1|o2|o3|o4|o5
    # pegar passageiro (qualquer um dos NPX que esteja neste no)
    hit = (m.po == m.no[:,None]); tem = hit.any(1); ks = hit.float().argmax(1)
    pega = pode & (~serv_ok) & (m.pd_<0) & tem
    m.po_ = torch.where(pega, m.no, m.po_)
    m.pd_ = torch.where(pega, m.pd.gather(1, ks[:,None]).squeeze(1), m.pd_)
    # respawn incondicional-mascarado (o sorteio roda p/ todos, so aplica onde pegou)
    if c.mundo in ('hotspot','mare'):
        novo_po = m._spawn_polo()
    else:
        novo_po = torch.randint(0,m.n,(B,),generator=m.gsim,device=DEV)
    novo_pd = torch.randint(0,m.n,(B,),generator=m.gsim,device=DEV)
    ks1 = ks[:,None]
    m.po.scatter_(1, ks1, torch.where(pega, novo_po,
                   m.po.gather(1,ks1).squeeze(1))[:,None])
    m.pd.scatter_(1, ks1, torch.where(pega, novo_pd,
                   m.pd.gather(1,ks1).squeeze(1))[:,None])
    # dinamica OCULTA do mundo
    if c.mundo == 'hotspot':
        # o polo deriva: com prob `deriva` por passo, pula para um vizinho
        # (vizinho uniforme = argmax de U(0,1) mascarado nos validos)
        rw = torch.rand(B, generator=m.gsim, device=DEV) < c.deriva
        vzs = m.Aa[m.cidx, m.hot]                              # (B,GMAX)
        u = torch.rand(B, GMAX, generator=m.gsim, device=DEV)
        pk = torch.where(vzs>=0, u, torch.full_like(u,-1.)).argmax(1)
        novo = vzs.gather(1, pk[:,None]).squeeze(1)
        m.hot = torch.where(rw & (novo>=0), novo, m.hot)
    elif c.mundo == 'mare':
        # dois polos trocam com periodo fixo; a FASE e oculta
        m.tmare += 1
        vira = m.tmare >= c.per_mare
        h = m.hot; m.hot = torch.where(vira, m.polo2, h)
        m.polo2 = torch.where(vira, h, m.polo2)
        m.tmare = torch.where(vira, torch.zeros_like(m.tmare), m.tmare)
    # mover
    mv = pode & (~serv_ok) & (~pega) & (esc<GMAX)
    viz = m.A_(); sel = esc.clamp(max=GMAX-1)
    dst = torch.gather(viz,1,sel[:,None]).squeeze(1)
    val = mv & (dst>=0)
    wru = torch.gather(m.W_(),1,sel[:,None]).squeeze(1)
    npd = torch.gather(m.npc,1,dst.clamp(min=0)[:,None]).squeeze(1)
    cu = wru + npd                       # custo em TEMPO (rest, fadiga)
    if c.mundo == 'ruas':
        # engarrafamento gasta TEMPO, nao tanque: fisico paga so 30% do jam.
        # (sem isto, cronica forte = 20+ de fuel por travessia = morte em serie
        #  — o teto fisico batia no chao antes de qualquer memoria poder pagar)
        cuf = wru + .3*npd
    else:
        cuf = cu
    sf = val & (m.fuel < cuf*c.fuel)
    m.avar += sf.float(); m.tpar = torch.where(sf, torch.full_like(m.tpar,18.), m.tpar)
    m.fuel = torch.where(sf, torch.full_like(m.fuel,15.), m.fuel)
    m.dinh = (m.dinh-12*sf.float()).clamp(min=0)
    rd = torch.rand(B, generator=m.gsim, device=DEV)
    fu = val & (~sf) & (rd < ((1-m.pneu/100)-.45).clamp(min=0)*c.risco*m.txp())
    m.avar += fu.float(); m.tpar = torch.where(fu, torch.full_like(m.tpar,15.), m.tpar)
    m.pneu = torch.where(fu, torch.full_like(m.pneu,60.), m.pneu)
    ol = val & (~sf) & (~fu) & (m.oleo<12) & (rd<.05)
    m.avar += ol.float(); m.tpar = torch.where(ol, torch.full_like(m.tpar,22.), m.tpar)
    m.oleo = torch.where(ol, torch.full_like(m.oleo,45.), m.oleo)
    qt = val & (~sf) & (~fu) & (~ol) & (m.temp>92)
    m.avar += qt.float(); m.tpar = torch.where(qt, torch.full_like(m.tpar,14.), m.tpar)
    m.temp = torch.where(qt, torch.full_like(m.temp,40.), m.temp)
    gs = torch.gather(m.grp_(),1,sel[:,None]).squeeze(1).float()
    fn = torch.gather(m.fase,1,m.no[:,None]).squeeze(1)
    vd = (gs==fn) | (m.tem_()==0)
    an = val & (~sf) & (~fu) & (~ol) & (~qt) & vd
    m.indo = torch.where(an, dst, m.indo)
    m.rest = torch.where(an, (cu*m.vel()).round().clamp(min=1), m.rest)
    m.fuel = m.fuel - cuf*c.fuel*an.float()
    m.fad  = (m.fad  + cu*c.fadiga*an.float()).clamp(max=100)
    m.pneu = (m.pneu - cuf*c.pneu*m.txp()*an.float()).clamp(min=0)
    m.oleo = (m.oleo - cuf*c.oleo*m.txo()*an.float()).clamp(min=0)
    m.temp = (m.temp + cuf*c.temp*m.txt()*an.float()).clamp(max=120)
    if c.mundo == 'ruas':
        # flutua EM TORNO da base cronica (AR-1): a estrutura persiste
        m.npc = (m.npc_base + (m.npc - m.npc_base)*.85
                 + (torch.rand(B,m.n,generator=m.gsim,device=DEV)-.5)*.20).clamp(min=0)
    else:
        m.npc = m.npc*.85 + torch.rand(B,m.n,generator=m.gsim,device=DEV)*(c.trafego*.45)
    return m

# ============================ CEREBROS ============================
# Contrato: extra() roda DENTRO do torch.compile — nao pode mutar tensor
# persistente IN-PLACE (scatter_/add_/index_put_ em atributo: o compile copia o
# tensor inteiro por passo — a diadica pagava 327 MB/passo por isso). REBIND
# (s.E = a*s.E + ...) e permitido: o dynamo rastreia setattr out-of-place.
# Mutacao in-place vive em atualizar(), que o loop chama FORA do grafo.
class Reativo:
    nome='reativo'; NP=NF+6+6+6   # I8: pesos mortos ate igualar 36
    def reset(s,m,W): pass
    def extra(s,m,W,viz): return 0.
    def atualizar(s,m): pass
class BancoEMA:
    nome='banco EMA'; NP=NF+6+6+6
    def reset(s,m,W):
        s.al = torch.exp(-1/(2+30*(W[:,NF+9:NF+12].abs()+.1)))   # (B,3)
        s.E  = torch.zeros(m.B,3,m.n,device=DEV)
    def extra(s,m,W,viz):
        oc = torch.zeros(m.B,m.n,device=DEV); oc.scatter_(1,m.no[:,None],1.)
        a = s.al[:,:,None]
        s.E = a*s.E + (1-a)*oc[:,None,:]
        g = s.E.gather(2, viz[:,None,:].expand(-1,3,-1))         # (B,3,G)
        return torch.einsum('bk,bkg->bg', W[:,NF+6:NF+9], g)
    def atualizar(s,m): pass

class _JanelasDiadicas:
    """Historico de nos em janelas [2^j, 2^(j+1)) via RING BUFFER uint8.

    A versao antiga materializava (B, 2^j, G) por nivel POR PASSO (335 MB em
    B=524k); uma versao intermediaria com contagens (B,L,n) + scatter_add
    pagava 5,8 ms/passo so de atomics espalhados. Aqui: o compile funde
    comparacao+peso+reducao sobre (B,64,G) sem materializar nada — 1,3 ms.
    A media da janela j = soma de igualdades x (1/largura), identica em valor
    (teste de equivalencia em src/testes.py). Contador e tensor no device
    (regra 3: int python vira guard/recompile)."""
    L=6; N=64
    def reset(s, m):
        assert m.n <= 255, 'ring uint8: n>255 faria mod-256 silencioso'
        s.buf = torch.zeros(m.B, s.N, dtype=torch.uint8, device=DEV)
        s.pt  = torch.zeros((), dtype=torch.long, device=DEV)
        s.ARL = torch.arange(s.N, device=DEV)
        # nivel de cada lag k (janela j cobre lags [2^j, 2^(j+1)))
        lvl = [0]*s.N
        for k in range(1, s.N): lvl[k] = min(k.bit_length()-1, s.L-1)
        s.LVL = torch.tensor(lvl, device=DEV)
        # WL[p,k] = 1/largura da janela do lag k quando ha p itens no historico
        # (0 se o lag ainda nao existe: k >= p — mesmo corte do slice antigo)
        WL = torch.zeros(s.N+1, s.N)
        for p in range(s.N+1):
            for k in range(1, s.N):
                j = min(k.bit_length()-1, s.L-1)
                a, b = 2**j, min(2**(j+1), s.N)
                w = max(min(b, p) - a, 0)
                if a <= k < min(b, p) and w > 0:
                    WL[p, k] = 1./w
        s.WL = WL.to(DEV)
    def saida(s, m, viz, Wj):
        """(B,G): soma ponderada das frequencias por janela, ja x pesos Wj (B,L)."""
        lagpos = (s.pt-1-s.ARL) % s.N                       # posicao do lag k no ring
        hist = s.buf.gather(1, lagpos[None,:].expand(m.B,-1))          # (B,N)
        # index_select (nao indexacao por tensor 0-dim, que faz .item() -> sync)
        wl = s.WL.index_select(0, s.pt.clamp(max=s.N).reshape(1))      # (1,N)
        coef = Wj.gather(1, s.LVL[None,:].expand(m.B,-1)) * wl         # (B,N)
        eq = (hist[:,:,None] == viz[:,None,:].to(torch.uint8)).float() # (B,N,G)
        return (coef[:,:,None]*eq).sum(1)
    def push_no(s, m, no):
        s.buf.scatter_(1, (s.pt % s.N).reshape(1,1).expand(m.B,1),
                       no.to(torch.uint8)[:,None])
        s.pt = s.pt + 1

class Diadica:
    nome='diadica'; NP=NF+6+6+6; L=6
    def reset(s,m,W):
        s.jan = _JanelasDiadicas(); s.jan.reset(m)
    def extra(s,m,W,viz):
        out = s.jan.saida(m, viz, W[:,NF+6:NF+6+s.jan.L])
        # o no a empurrar e o de ANTES do passo (m.no e re-atribuido, nao mutado,
        # entao guardar a referencia preserva o valor pre-passo)
        s._no = m.no
        return out
    def atualizar(s,m):
        s.jan.push_no(m, s._no)
class Corpo:
    nome='corpo'; NP=NF+6+6+6
    def reset(s,m,W): pass
    def atualizar(s,m): pass
    def extra(s,m,W,viz):
        d = torch.stack([1-m.fuel/m.capf(), m.fome/100, m.fad/100,
                         1-m.pneu/100, m.temp/120],1)
        k = d.argmax(1); mp = m.MP[k]
        sel = m.dsa[m.cidx, mp].gather(1, viz)                   # (B,G)
        peso = W[m.ARB, NF+6+k]
        return (peso*d.max(1).values)[:,None]*(-sel/6.)
class Hibrido:
    nome='hibrido'; NP=NF+6+6+6; L=6
    def reset(s,m,W):
        s.jan = _JanelasDiadicas(); s.jan.reset(m)
    def extra(s,m,W,viz):
        out = s.jan.saida(m, viz, W[:,NF+6:NF+6+s.jan.L])
        s._no = m.no
        d = torch.stack([1-m.fuel/m.capf(), m.fome/100, m.fad/100,
                         1-m.pneu/100, m.temp/120],1)
        k = d.argmax(1); mp = m.MP[k]
        sel = m.dsa[m.cidx, mp].gather(1, viz)
        peso = W[m.ARB, NF+12+k]
        return out + (peso*d.max(1).values)[:,None]*(-sel/6.)
    def atualizar(s,m):
        s.jan.push_no(m, s._no)
class MemCusto:
    """EMA de congestao POR NO, escrita ao pisar, lida nos vizinhos.
    E a memoria que o mundo `ruas` exige: la a congestao dos vizinhos e
    invisivel e a estrutura e cronica — quem lembra desvia."""
    nome='mem custo'; NP=NF+6+6+6
    def reset(s,m,W):
        s.k = torch.sigmoid(W[:,NF+6:NF+7])          # taxa de escrita evoluida
        s.C = torch.zeros(m.B, m.n, device=DEV)      # congestao lembrada por no
    def extra(s,m,W,viz):
        s._obs = torch.gather(m.npc, 1, m.no[:,None])   # sentida AGORA, aqui
        s._no = m.no
        g = torch.gather(s.C, 1, viz)
        return W[:,NF+7:NF+8]*(-g)                   # evita vizinho lembrado como ruim
    def atualizar(s,m):
        atual = torch.gather(s.C, 1, s._no[:,None])
        s.C.scatter_(1, s._no[:,None], atual + s.k*(s._obs-atual))
class MapaServico:
    """Mapa APRENDIDO de servicos: grava onde ja viu posto/comida/etc ao passar,
    e puxa para o vizinho lembrado quando a necessidade aperta. E o `corpo` sem
    oraculo — a memoria que o mundo `semgps` exige."""
    nome='mapa servico'; NP=NF+6+6+6
    def reset(s,m,W):
        s.M = torch.zeros(m.B, 5, m.n, device=DEV)   # 1 = "ja vi este servico aqui"
    def extra(s,m,W,viz):
        s._sv = m.serv_(); s._no = m.no
        d = torch.stack([1-m.fuel/m.capf(), m.fome/100, m.fad/100,
                         1-m.pneu/100, m.temp/120],1)
        k = d.argmax(1); mp = m.MP[k]
        sel = s.M[m.ARB, mp].gather(1, viz)
        peso = W[m.ARB, NF+12+k]
        return (peso*d.max(1).values)[:,None]*sel
    def atualizar(s,m):
        no3 = s._no[:,None,None].expand(-1,5,1)
        cur = s.M.gather(2, no3)
        s.M.scatter_(2, no3, torch.maximum(cur, s._sv[:,:,None]))
ARQS = [Reativo, BancoEMA, Diadica, Corpo, Hibrido, MemCusto, MapaServico]

# ============================ RODAR / EVOLUIR ============================
def um_passo(m, a, Wb):
    """um tick completo: observar -> pontuar -> agir."""
    F, val = m.obs(); viz = m.A_().clamp(min=0)
    sc = torch.einsum('bgf,bf->bg', F, Wb[:,:NF]) + a.extra(m, Wb, viz)
    sc = sc.masked_fill(~val, -1e9)
    svn, sv = m.serv_need()
    ss = (svn*Wb[:,NF:NF+5] + Wb[:,NF+5:NF+6]).masked_fill(sv<=0, -1e9)
    passo(m, torch.cat([sc, ss],1).argmax(1))

# torch.compile e opcional (NM_COMPILE=0 desliga). Fallback silencioso p/ eager
# se o backend (triton) nao suportar a GPU. O ganho vem da fusao dos ~200
# kernels elementwise por passo.
_PASSO_C = None
_COMPILE_OK = None
def _passo_fn():
    global _PASSO_C, _COMPILE_OK
    if os.environ.get('NM_COMPILE', '1') == '0' or DEV != 'cuda':
        return um_passo
    if _COMPILE_OK is False:
        return um_passo
    if _PASSO_C is None:
        try:
            import torch._dynamo
            torch._dynamo.config.cache_size_limit = 64
            _PASSO_C = torch.compile(um_passo, dynamic=False)
        except Exception:
            _COMPILE_OK = False
            return um_passo
    return _PASSO_C

def modo_execucao():
    """'compile' ou 'eager' — gravar no JSON: compile e eager divergem por
    arredondamento (~1e-5 em fuel/temp), entao bateria com modo misto precisa
    ficar auditavel."""
    if os.environ.get('NM_COMPILE', '1') == '0' or DEV != 'cuda' or _COMPILE_OK is False:
        return 'eager'
    return 'compile'

def rodar(Arq, W, cidades, cfg, T, seed, R=1):
    global _COMPILE_OK
    P = W.shape[0]; m = Lote(cidades, P, cfg, seed=seed, R=R)
    Wb = W[m.gidx]
    a = Arq(); a.reset(m, Wb)
    fn = _passo_fn()
    for t in range(T):
        try:
            fn(m, a, Wb)
        except Exception as e:
            # compile falha na 1a chamada de um grafo novo (arq/shape novos = t==0
            # desta rodada). O passo pode ter executado PELA METADE (subgrafos
            # anteriores ja mutaram m) — nunca re-executar por cima: reconstruir
            # a rodada inteira do zero em eager.
            if fn is not um_passo and t == 0:
                print(f'  [compile falhou ({type(e).__name__}); refazendo esta '
                      f'rodada em eager — modo gravado no JSON]', flush=True)
                _COMPILE_OK = False
                return rodar(Arq, W, cidades, cfg, T, seed, R=R)
            raise RuntimeError(
                f'passo falhou no meio da rodada (t={t}); estado possivelmente '
                f'parcial. Se foi torch.compile, rode com NM_COMPILE=0 — a '
                f'retomada por checkpoint e segura.') from e
        a.atualizar(m)     # mutacao de estado da memoria FORA do grafo compilado
    return m

def avaliar(Arq, W, cidades, cfg, T, seed=0, R=1):
    m = rodar(Arq, W, cidades, cfg, T, seed, R=R)
    return (m.ent - 0.3*m.avar).view(W.shape[0], m.NC*m.R).mean(1)

def evoluir(Arq, cidades, cfg, P=512, ger=25, T=1500, seed=0, elite=.15,
            verbose=True, R=1, ckpt=None, ckpt_cada=10):
    """ckpt: caminho .pt — salva populacao+RNG a cada `ckpt_cada` geracoes e
    RETOMA automaticamente se o arquivo existir (maquina pode cair no meio)."""
    g = torch.Generator(device=DEV); g.manual_seed(seed)
    W = torch.randn(P, Arq.NP, generator=g, device=DEV)*.8
    hist, snaps, it0 = [], [], 0
    # impressao digital da config: retomar com config DIFERENTE misturaria
    # populacoes/geracoes de corridas distintas num JSON que declara uma so
    cfg_id = f'P={P} ger={ger} T={T} R={R} seed={seed} elite={elite} NP={Arq.NP} arq={Arq.nome}'
    if ckpt and os.path.exists(ckpt):
        d = torch.load(ckpt, map_location=DEV, weights_only=True)
        if d.get('cfg_id') != cfg_id:
            raise SystemExit(
                f'CHECKPOINT DE OUTRA CONFIG:\n  ckpt:  {d.get("cfg_id")}\n'
                f'  agora: {cfg_id}\n  Use --do-zero para descartar, ou repita a config original.')
        W = d['W']; hist = d['hist']; snaps = d['snaps']; it0 = d['ger']+1
        g.set_state(d['rng'].cpu())
        if verbose:
            aviso = ' (evolucao ja concluida — vai direto para avaliacao)' if it0 >= ger else ''
            print(f'  [retomando {ckpt} da geracao {it0}{aviso}]')
    for it in range(it0, ger):
        f = avaliar(Arq, W, cidades, cfg, T, seed+it*13, R=R)
        o = f.argsort(descending=True); W = W[o]; f = f[o]
        hist.append(float(f[0]))
        ne = max(2, int(P*elite)); pais = W[:ne]
        idx = torch.randint(0, ne, (P-ne,), generator=g, device=DEV)
        fil = (pais[idx]+torch.randn(P-ne,Arq.NP,generator=g,device=DEV)*.3).clamp(-5,5)
        Wn = torch.cat([pais, fil])
        if it % 5 == 0:
            # snapshot da POPULACAO, nao so do melhor. Sem isto nao da para
            # distinguir "a populacao aprendeu" de "um sortudo puxou a curva" —
            # que foi exatamente o modo de falha do curriculo por melhor-individuo.
            q = torch.tensor([0., .25, .5, .75, 1.], device=DEV)
            snaps.append({
                'ger': it,
                'fit_percentis': [float(x) for x in torch.quantile(f, q)],
                'fit_media': float(f.mean()),
                # diversidade = desvio medio dos pesos na populacao. Se cai a
                # zero, virou monocultura e a evolucao parou de explorar.
                'diversidade': float(Wn.std(0).mean()),
                'diversidade_elite': float(Wn[:max(2, int(P*elite))].std(0).mean()),
            })
        W = Wn
        if ckpt and (it % ckpt_cada == 0 or it == ger-1):
            tmp = ckpt + '.tmp'
            torch.save({'W': W, 'ger': it, 'hist': hist, 'snaps': snaps,
                        'rng': g.get_state(), 'cfg_id': cfg_id}, tmp)
            os.replace(tmp, ckpt)      # troca atomica: nunca fica ckpt corrompido
        if verbose and it % 5 == 0:
            vram = torch.cuda.max_memory_allocated()/1e9 if DEV=='cuda' else 0
            print(f'  ger {it:>3}  melhor {float(f[0]):>7.2f}  media {float(f.mean()):>7.2f}'
                  f'  div {snaps[-1]["diversidade"]:.3f}  VRAM {vram:.1f} GB', flush=True)
    return W[0], hist, snaps


In [ ]:
%%writefile src/rodar.py
#!/usr/bin/env python3
"""Bateria de taxis. Roda igual em CPU, GPU local ou Colab.

    python src/rodar.py --smoke               # teste de fumaca (segundos)
    python src/rodar.py --perito              # PORTAO de validacao (obrigatorio)
    python src/rodar.py --bateria             # bateria completa
    python src/rodar.py --bateria -P 8192 -G 100 -R 8
"""
import argparse, json, os, sys, time
from datetime import datetime
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch, numpy as np
from lab import (DEV, GMAX, NPX, NF, gerar_cidade, Cfg, Lote, passo,
                 Reativo, BancoEMA, Diadica, Corpo, Hibrido, ARQS,
                 rodar, avaliar, evoluir, um_passo, modo_execucao)

NOMES_F = (['d_alvo','custo_rua','congestao','semaforo',
            'd_posto','d_comida','d_descanso','d_oficina','d_loja',
            'e_fuel','e_fome','e_fadiga','e_pneu','e_temp','e_oleo',
            'e_dinheiro','e_tem_pax','bias']
           + ['sv_posto','sv_comida','sv_descanso','sv_oficina','sv_loja','sv_bias']
           + [f'mem_{i}' for i in range(6)] + [f'corpo_{i}' for i in range(6)])


def perito_W(NP):
    """pesos escritos a mao — referencia humana. Memoria fica ZERADA de proposito."""
    w = torch.zeros(NP, device=DEV)
    w[0]=4.0; w[1]=0.6; w[2]=0.8; w[3]=1.2
    w[4]=1.2; w[5]=.10; w[6]=.05; w[7]=.60; w[8]=.05
    w[NF:NF+5] = torch.tensor([20.,14.,10.,18.,3.], device=DEV)
    w[NF+5] = -4.0
    return w


def limpar():
    import gc
    for _ in range(2):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def smoke(mundo='base'):
    print('--- teste de fumaca ---')
    cids = [gerar_cidade(s) for s in range(2)]
    cfg = Cfg(1.0, mundo=mundo)
    for A in ARQS:
        W = torch.randn(4, A.NP, device=DEV)*.8
        m = rodar(A, W, cids, cfg, T=150, seed=0)
        print(f'  {A.nome:<12} entregas {m.ent.mean():5.2f}  B={m.B}  OK')
    print('ok')


def portao(mundo='base', T=1200, n_cid=3, rep=16, R=2):
    """PORTAO OBRIGATORIO.

    Com os pesos de memoria zerados, as 5 arquiteturas sao a MESMA politica.
    Elas TEM que produzir resultado identico. Se divergirem, ha bug de porte
    (NaN silencioso, indice fora de faixa, gerador consumido em ordem diferente).
    Este teste pegou 5 bugs durante a migracao CPU -> GPU.
    """
    print('--- portao de validacao ---')
    cids = [gerar_cidade(s) for s in range(n_cid)]
    cfg = Cfg(1.0, mundo=mundo)
    print(f"  {'arquitetura':<12}{'PERITO':>9}{'aleatorio':>11}{'razao':>8}{'avarias':>9}")
    vals = []
    for A in ARQS:
        Wp = perito_W(A.NP)[None,:].repeat(rep,1)
        mp = rodar(A, Wp, cids, cfg, T=T, seed=1, R=R)
        Wr = torch.randn(rep, A.NP, device=DEV)*.8
        mr = rodar(A, Wr, cids, cfg, T=T, seed=1, R=R)
        ep, er = mp.ent.mean().item(), mr.ent.mean().item()
        av = mp.avar.mean().item()
        vals.append(ep)
        print(f'  {A.nome:<12}{ep:>9.1f}{er:>11.1f}{ep/max(er,.05):>7.1f}x{av:>9.1f}')
        limpar()
    ok = (max(vals) - min(vals)) < 0.05
    print(f"\n  {'PASSOU' if ok else 'FALHOU'}: as {len(ARQS)} "
          f"{'batem' if ok else 'DIVERGEM'} (spread {max(vals)-min(vals):.3f})")
    if not ok:
        print('  >>> bug de porte. NAO rode a bateria.')
    return ok


def paisagem(Arq, w, cidades, cfg, T=400, R=2, passos=9, raio=1.5):
    """Perturba CADA peso em 9 pontos -> revela os VALES e PICOS de f(w).
    Tudo num batch so: (NP*9) genomas avaliados de uma vez."""
    NPw = w.shape[0]
    d = torch.linspace(-raio, raio, passos, device=DEV)
    Wp = w[None,None,:].repeat(NPw, passos, 1)
    ii = torch.arange(NPw, device=DEV)[:,None].expand(-1,passos)
    jj = torch.arange(passos, device=DEV)[None,:].expand(NPw,-1)
    Wp[ii, jj, ii] = w[ii] + d[None,:]
    f = avaliar(Arq, Wp.reshape(NPw*passos, NPw), cidades, cfg, T, seed=555, R=R)
    return dict(deltas=[round(x,3) for x in d.tolist()],
                curvas=[[round(x,4) for x in l] for l in f.view(NPw,passos).tolist()])


def trajetoria(Arq, w, cidades, cfg, T=1500, amostras=200):
    """Serie temporal de UM mundo: como o corpo e o dinheiro evoluem."""
    m = Lote(cidades[:1], 1, cfg, seed=4242, R=1)
    Wb = w[None,:].repeat(m.B,1)
    a = Arq(); a.reset(m, Wb)
    pa = max(1, T//amostras); buf = []
    for t in range(T):
        um_passo(m, a, Wb); a.atualizar(m)
        if t % pa == 0:
            buf.append(torch.stack([m.fuel[0], m.fome[0], m.fad[0], m.pneu[0],
                                    m.temp[0], m.oleo[0], m.dinh[0], m.ent[0],
                                    m.avar[0], m.no[0].float(),
                                    (m.pd_[0] >= 0).float()]))
    M = torch.stack(buf)
    ks = ['fuel','fome','fadiga','pneu','temp','oleo','dinheiro',
          'entregas','avarias','no','tem_pax']
    return {k: [round(x,2) for x in M[:,i].tolist()] for i,k in enumerate(ks)}


def salvar_json(caminho, obj):
    """escrita atomica: escreve .tmp e troca — queda de maquina nunca corrompe."""
    tmp = caminho + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f)
    os.replace(tmp, caminho)


def bateria(P, GER, R, T_EVO, T, n_treino, n_teste, seeds, saida_dir, seed_evo=0,
            mundo='base', do_zero=False, n_cidade=26):
    treino = [gerar_cidade(s, n=n_cidade) for s in range(n_treino)]
    teste  = [gerar_cidade(1000+s, n=n_cidade) for s in range(n_teste)]
    cfg = Cfg(1.0, trafego=.35, imposto=.18, mundo=mundo)
    print(f'mundos/geracao: {P*n_treino*R:,} | por genoma: {n_treino*R}')

    # --- retomada: resultado parcial por arquitetura + checkpoint por geracao ---
    os.makedirs(saida_dir, exist_ok=True)
    parcial = os.path.join(saida_dir, f'resultado-taxi-{mundo}-s{seed_evo}-PARCIAL.json')
    ckpt_de = lambda nome: os.path.join(
        saida_dir, f'ckpt-{mundo}-s{seed_evo}-{nome.replace(" ","_")}.pt')
    cfg_id = (f'P={P} G={GER} R={R} tevo={T_EVO} taval={T} treino={n_treino} '
              f'teste={n_teste} seeds={seeds} mundo={mundo} seed_evo={seed_evo} '
              f'n={n_cidade}')
    res = {}
    if do_zero:
        for f in [parcial] + [ckpt_de(A.nome) for A in ARQS]:
            if os.path.exists(f): os.remove(f)
    elif os.path.exists(parcial):
        d = json.load(open(parcial))
        if d.get('config_str') != cfg_id:
            sys.exit(f'PARCIAL DE OUTRA CONFIG:\n  parcial: {d.get("config_str")}\n'
                     f'  agora:   {cfg_id}\n'
                     f'  Use --do-zero para descartar, ou repita a config original.')
        res = d.get('resultados', {})
        if res: print(f'[retomando: {list(res.keys())} ja prontos em {parcial}]')

    if torch.cuda.is_available():
        limpar(); torch.cuda.reset_peak_memory_stats()
        _ = rodar(Hibrido, torch.randn(P, Hibrido.NP, device=DEV),
                  treino, cfg, T=5, seed=0, R=R)
        pico = torch.cuda.max_memory_allocated()/1e9
        tot = torch.cuda.get_device_properties(0).total_memory/1e9
        print(f'memoria: {pico:.2f} de {tot:.1f} GB ({pico/tot*100:.0f}%)')
        del _; limpar()
        if pico > tot*0.8:
            sys.exit(f'REDUZA P (sugestao {int(P*tot*0.7/pico)})')
        print(f'cabe ate P ~= {int(P*tot*0.7/max(pico,1e-6)):,}')

    t0 = time.time()
    for Arq in ARQS:
        if Arq.nome in res:
            print(f'\n=== {Arq.nome}: ja pronto, pulando ===')
            continue
        print(f'\n=== {Arq.nome} ({Arq.NP} params) ===')
        ta = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        w, hist, snaps = evoluir(Arq, treino, cfg, P, GER, T_EVO, R=R,
                                seed=seed_evo, ckpt=ckpt_de(Arq.nome))

        m = rodar(Arq, w[None,:].repeat(seeds,1), teste, cfg, T, seed=7777, R=1)
        ent, avr, ups = m.ent, m.avar, m.up
        rec, imp, din = m.rec, m.imp, m.dinh
        q = torch.tensor([0,.10,.25,.50,.75,.90,1.0], device=DEV)
        lo, hi = float(ent.min()), float(ent.max())
        hb = torch.histc(ent, bins=20, min=lo, max=max(hi, lo+1e-3))
        est = dict(
            params=int(Arq.NP), hist=hist, snapshots=snaps,
            pesos_finais=[round(x,4) for x in w.tolist()],
            nomes_pesos=NOMES_F[:Arq.NP],
            entregas=dict(media=round(float(ent.mean()),3),
                          dp=round(float(ent.std()),3),
                          percentis=[round(x,2) for x in torch.quantile(ent,q).tolist()],
                          histograma=[int(x) for x in hb.tolist()],
                          hist_min=round(lo,2), hist_max=round(hi,2)),
            avarias=dict(media=round(float(avr.mean()),3),
                         dp=round(float(avr.std()),3)),
            upgrades=dict(media=round(float(ups.sum(1).mean()),3),
                          por_tipo=[round(x,3) for x in ups.mean(0).tolist()]),
            receita=dict(media=round(float(rec.mean()),2),
                         imposto=round(float(imp.mean()),2),
                         dinheiro_final=round(float(din.mean()),2)),
            n_amostras=int(ent.numel()))
        del m; limpar()

        print('  paisagem...')
        est['paisagem'] = paisagem(Arq, w, treino[:2], cfg); limpar()
        print('  trajetoria...')
        est['trajetoria'] = trajetoria(Arq, w, teste, cfg, T=T); limpar()

        est['tempo_s'] = round(time.time()-ta, 1)
        est['vram_pico_gb'] = (round(torch.cuda.max_memory_allocated()/1e9, 2)
                               if torch.cuda.is_available() else 0)
        est['modo'] = modo_execucao()   # compile e eager divergem ~1e-5: auditar
        res[Arq.nome] = est
        # salva o PARCIAL agora: se a maquina cair, o que ja rodou esta no disco
        salvar_json(parcial, dict(data=datetime.now().isoformat(), parcial=True,
                                  config_str=cfg_id, resultados=res))
        ck = ckpt_de(Arq.nome)
        if os.path.exists(ck): os.remove(ck)   # evolucao desta arq concluida
        print(f"  -> {est['entregas']['media']:.1f} +- {est['entregas']['dp']:.1f} "
              f"entregas ({est['n_amostras']} amostras, {est['tempo_s']:.0f}s) "
              f"[parcial salvo]")

    print(f'\nTOTAL: {time.time()-t0:.0f}s')
    out = dict(
        data=datetime.now().isoformat(),
        config=dict(P=P, R=R, geracoes=GER, T_evo=T_EVO, T_aval=T,
                    cidades_treino=n_treino, cidades_teste=n_teste, seeds=seeds,
                    mundos_por_geracao=P*n_treino*R, mundos_por_genoma=n_treino*R,
                    dificuldade=1.0, trafego=.35, imposto=.18),
        dispositivo=(torch.cuda.get_device_name(0)
                     if torch.cuda.is_available() else 'cpu'),
        resultados=res)
    cam = os.path.join(saida_dir, f'resultado-taxi-{mundo}-s{seed_evo}-{datetime.now():%Y%m%d-%H%M}.json')
    salvar_json(cam, out)
    if os.path.exists(parcial): os.remove(parcial)
    print('salvo:', cam, '|', round(os.path.getsize(cam)/1e6, 2), 'MB')
    print('\nRESUMO:')
    for k, v in sorted(res.items(), key=lambda x: -x[1]['entregas']['media']):
        print(f"  {k:<12}{v['entregas']['media']:>7.1f} +- "
              f"{v['entregas']['dp']:>5.1f}  avarias {v['avarias']['media']:>5.1f}  "
              f"upg {v['upgrades']['media']:>4.1f}  ({v['params']} par)")
    return cam


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('--smoke',   action='store_true', help='teste rapido')
    ap.add_argument('--perito',  action='store_true', help='portao de validacao')
    ap.add_argument('--bateria', action='store_true', help='bateria completa')
    ap.add_argument('-P', type=int, default=2048, help='populacao')
    ap.add_argument('-G', '--geracoes', type=int, default=60)
    ap.add_argument('-R', type=int, default=8, help='replicas por genoma')
    ap.add_argument('--tevo',  type=int, default=600)
    ap.add_argument('--taval', type=int, default=1500)
    ap.add_argument('--treino', type=int, default=8)
    ap.add_argument('--teste',  type=int, default=6)
    ap.add_argument('--seeds',  type=int, default=512)
    # I4: a evolucao rodava SEMPRE com seed 0 — uma unica corrida evolutiva por
    # arquitetura. Rode a bateria com --seed-evo 0/1/2 e compare: diferenca
    # entre sementes e o ruido de base; efeito menor que isso nao e efeito.
    ap.add_argument('--seed-evo', type=int, default=0, dest='seed_evo')
    ap.add_argument('--mundo', default='base',
                    choices=['base','cego','hotspot','mare','ruas','semgps'])
    ap.add_argument('--do-zero', action='store_true', dest='do_zero',
                    help='ignora parcial/checkpoint existente e recomeca')
    ap.add_argument('--n-cidade', type=int, default=26, dest='n_cidade',
                    help='nos por cidade (max 255; cidades maiores = rotas longas)')
    ap.add_argument('--out', default=os.path.join(
        os.path.dirname(os.path.dirname(os.path.abspath(__file__))), 'resultados'))
    a = ap.parse_args()

    print(f'dispositivo: {DEV}')
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0),
              f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

    if a.smoke:
        smoke()
    if a.perito:
        portao(a.mundo)
    if a.bateria:
        if not portao(a.mundo):
            sys.exit('portao falhou — corrija antes de rodar a bateria')
        bateria(a.P, a.geracoes, a.R, a.tevo, a.taval,
                a.treino, a.teste, a.seeds, a.out, seed_evo=a.seed_evo,
                mundo=a.mundo, do_zero=a.do_zero, n_cidade=a.n_cidade)
    if not (a.smoke or a.perito or a.bateria):
        ap.print_help()


## 1. Portão de validação — obrigatório

`PASSOU: as 5 batem (spread 0.000)` ou **pare aqui**.

In [ ]:
!python src/rodar.py --perito

## 2. Bateria — 3 sementes de evolução

T4 16 GB aguenta P=8192. Cada semente gera um JSON próprio no Drive.

**Sessão caiu no meio?** Rode esta célula de novo: retoma sozinha (parcial por
arquitetura + checkpoint de evolução a cada 10 gerações, tudo no Drive).
Para recomeçar do zero de propósito: adicione `--do-zero`.

In [ ]:
MUNDO = 'base'   # base | cego | hotspot | mare
for sd in (0, 1, 2):
    print(f'\n########## SEMENTE {sd} ##########')
    !python src/rodar.py --bateria -P 8192 -G 100 -R 8 --seed-evo {sd} --mundo {MUNDO} --out {OUT}

## 3. Leitura rápida do resultado

Entregas por arquitetura e semente + a régua de ruído (amplitude entre sementes).

In [ ]:
import json, glob, collections, os, re
docs = {}
for f in sorted(glob.glob(OUT + '/resultado-taxi-*.json')):
    base = os.path.basename(f)
    m = re.match(r'resultado-taxi-([a-z]+)-s(\d+)-', base)
    chave = m.groups() if m else (base,)
    d = json.load(open(f))
    if chave in docs and d.get('parcial'):   # final ganha do PARCIAL da mesma semente
        continue
    docs[chave] = d
por_arq = collections.defaultdict(list)
for chave, d in docs.items():
    for nome, r in d.get('resultados', {}).items():
        por_arq[nome].append(r['entregas']['media'])
    print('-'.join(chave), '(PARCIAL)' if d.get('parcial') else '')
for nome, v in sorted(por_arq.items(), key=lambda x: -sum(x[1])/len(x[1])):
    med = sum(v)/len(v)
    print(f"{nome:<12} media {med:5.2f}  sementes {['%.2f'%x for x in v]}  amplitude {max(v)-min(v):.2f}")

## 4. Limpar GPU após interromper

In [ ]:
import gc, torch
for _ in range(3):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
print(f'{torch.cuda.memory_allocated()/1e9:.2f} GB alocados')